# NikaMoE analysis

Trains the 2-expert MoE model and a dense (`n_experts=1`) baseline of the same shape on the two subject-matter corpora (`prose`, `code`), then checks:

1. **Per-domain loss**: does MoE beat the dense baseline on held-out loss for each domain?
2. **Routing specialization**: does the router send `prose` tokens and `code` tokens to different experts?

This notebook only calls into `nika_ai` and plots results — the model, training loop, and routing analysis all live in the package, not here.

In [ ]:
import matplotlib.pyplot as plt

from nika_ai.model import NikaMoEConfig
from nika_ai.train import TrainConfig, train
from nika_ai.data import DOMAINS
from nika_ai.inference import analyze_routing, expert_usage_fraction
import torch

## Train the MoE model (2 experts) and a dense baseline (1 expert)

The dense baseline reuses the exact same `MoELayer` code with `n_experts=1` — with one expert, routing always picks that expert, `MoELayer` degenerates into a plain SwiGLU MLP, and the load-balancing loss becomes a constant. This gives a fair architectural comparison for free instead of a second hand-written model.

In [ ]:
base_kwargs = dict(
    vocab_size=4000,
    hidden_size=128,
    n_layers=4,
    n_heads=4,
    expert_intermediate_size=2688,
    max_seq_len=128,
)

moe_config = NikaMoEConfig(n_experts=2, **base_kwargs)
dense_config = NikaMoEConfig(n_experts=1, **base_kwargs)

train_config = TrainConfig(max_steps=400, eval_interval=20, log_interval=400, checkpoint_dir="/tmp/nika_analysis_moe")
dense_train_config = TrainConfig(max_steps=400, eval_interval=20, log_interval=400, checkpoint_dir="/tmp/nika_analysis_dense")

moe_history, dense_history = [], []
model_moe, tokenizer, domain_splits = train(moe_config, train_config, history=moe_history)
model_dense, _, _ = train(dense_config, dense_train_config, history=dense_history)

## Held-out loss per domain: MoE vs. dense baseline

In [ ]:
fig, axes = plt.subplots(1, len(DOMAINS), figsize=(10, 4))
for ax, domain in zip(axes, DOMAINS):
    ax.plot([h["step"] for h in moe_history], [h[domain] for h in moe_history], label="MoE (2 experts)")
    ax.plot([h["step"] for h in dense_history], [h[domain] for h in dense_history], label="Dense (1 expert)")
    ax.set_title(f"{domain} val loss")
    ax.set_xlabel("step")
    ax.set_ylabel("cross-entropy loss")
    ax.legend()
fig.tight_layout()
plt.show()

## Routing specialization: which expert handles which domain?

For each domain's held-out text, run the MoE model with `collect_routing=True` and look at what fraction of tokens each layer sent to expert 0 vs. expert 1. If the router specialized by subject matter, the two domains' bars should look different — ideally close to mirror images of each other.

In [ ]:
device = "cpu"
usage_by_domain = {}
for domain in DOMAINS:
    # Truncate to max_seq_len: the raw val token stream can be longer than the
    # model's RoPE cache (precomputed up to config.max_seq_len=128), and this is
    # just a specialization sanity check, not a full-corpus evaluation.
    val_ids = domain_splits[domain].val.token_ids[: moe_config.max_seq_len]
    input_ids = torch.from_numpy(val_ids).unsqueeze(0)
    routing = analyze_routing(model_moe, input_ids, device)
    usage_by_domain[domain] = expert_usage_fraction(routing, moe_config.n_experts)

usage_by_domain

In [ ]:
n_layers = moe_config.n_layers
fig, axes = plt.subplots(1, n_layers, figsize=(4 * n_layers, 4), sharey=True)
width = 0.35
x = range(moe_config.n_experts)

for layer_idx, ax in enumerate(axes):
    for offset, domain in zip((-width / 2, width / 2), DOMAINS):
        usage = usage_by_domain[domain][layer_idx]
        ax.bar([xi + offset for xi in x], usage, width=width, label=domain)
    ax.set_xticks(list(x))
    ax.set_xticklabels([f"expert {e}" for e in x])
    ax.set_title(f"layer {layer_idx}")
    ax.set_ylabel("fraction of tokens routed here")
axes[0].legend()
fig.suptitle("Expert usage by domain, per layer")
fig.tight_layout()
plt.show()

## Reading the results

- If MoE's per-domain val loss beats the dense baseline's, the extra expert capacity is paying for itself.
- If the two domains' bar charts diverge in the same layer (e.g. `prose` → mostly expert 1, `code` → mostly expert 0), that layer's router has specialized by subject matter. Earlier layers tend to specialize more clearly than later ones, since later layers operate on representations that have already been shaped by the earlier routing decisions.
- This is a toy-scale demo (~9M total params -- 4 layers x 2 experts of ~1M each, plus attention/embeddings -- trained on a few KB of text): treat the exact numbers as illustrative, not as evidence of a general MoE result. The mechanism being exercised — sparse dispatch, the load-balancing loss, KV-cache-consistent routing — is the real, reusable part.